# Boston Vision Zero: Clean Crash Records

Fetches the City of Boston Vision Zero crash and fatality records from the Boston Open Data CKAN API, keeps records from 2021 onward, and normalizes them to the Mass Crash Map column names.

Output: `boston-vision-zero.csv` in the notebook's working directory. This is a standalone Vision Zero export; it does not merge or deduplicate against MassDOT.

## Data and interpretation

The crash dataset uses `dispatch_ts`; the fatality dataset uses `date_time`. Crash records are assigned severity 2 (injury crash) and fatality records severity 1, following the existing Boston City cleaner. `mode_type` sets the cyclist and pedestrian flags; police, other-user classification, and interstate status are left blank when this source does not provide them.

In [7]:
%pip install pandas requests

Note: you may need to restart the kernel to use updated packages.


In [8]:
from pathlib import Path

import pandas as pd
import requests

PACKAGE_URL = "https://data.boston.gov/api/3/action/package_show"
DATASTORE_URL = "https://data.boston.gov/api/3/action/datastore_search"
START_YEAR = 2021
PAGE_SIZE = 5000
REQUEST_TIMEOUT = 60
OUTPUT_FILE = Path("boston-vision-zero.csv")

## Retrieve the source tables

The helper discovers the active CKAN datastore resource and fetches every page. It raises an error if the API reports a failure or no active resource is available.

In [9]:
def fetch_dataset(dataset_id: str) -> pd.DataFrame:
    package_response = requests.get(
        PACKAGE_URL, params={"id": dataset_id}, timeout=REQUEST_TIMEOUT
    )
    package_response.raise_for_status()
    package_payload = package_response.json()
    if not package_payload.get("success"):
        raise RuntimeError(f"CKAN could not find dataset {dataset_id!r}")

    resources = package_payload["result"].get("resources", [])
    resource = next((item for item in resources if item.get("datastore_active")), None)
    if resource is None:
        raise RuntimeError(f"No active datastore resource for {dataset_id!r}")

    records = []
    offset = 0
    while True:
        response = requests.get(
            DATASTORE_URL,
            params={"resource_id": resource["id"], "limit": PAGE_SIZE, "offset": offset},
            timeout=REQUEST_TIMEOUT,
        )
        response.raise_for_status()
        payload = response.json()
        if not payload.get("success"):
            raise RuntimeError(f"CKAN query failed for dataset {dataset_id!r}")
        page = payload["result"].get("records", [])
        records.extend(page)
        if len(page) < PAGE_SIZE:
            break
        offset += PAGE_SIZE

    return pd.DataFrame.from_records(records)

In [10]:
crash_raw = fetch_dataset("vision-zero-crash-records")
fatality_raw = fetch_dataset("vision-zero-fatality-records")

print(f"Vision Zero crash records: {len(crash_raw):,}")
print(f"Vision Zero fatality records: {len(fatality_raw):,}")
display(crash_raw.head())
display(fatality_raw.head())

Vision Zero crash records: 44,669
Vision Zero fatality records: 162


,_id,dispatch_ts,mode_type,location_type,street,xstreet1,xstreet2,x_cord,y_cord,lat,long
0,1,2015-12-23 02:25:08+00,ped,Street,MASSACHUSETTS AVE,NEWMARKET SQ,ALLSTATE RD,773665.54,2944291.83,42.32610309166405,-71.06628057217974
1,2,2021-03-19 20:36:27+00,mv,Street,TURQUOISE WAY,HORADAN WAY,MCGREEVEY WAY,765042.52,2947002.83,42.334019355974135,-71.09732362405403
2,3,2019-12-17 17:51:34+00,mv,Street,HUNTINGTON AVE,PARKER HILL AVE,COLBURN ST,761401.35,2946426.56,42.332656898725865,-71.11099338207516
3,4,2017-01-18 12:48:27+00,ped,Street,FRANKLIN HILL AVE,FERMOY HEIGHTS AVE,AMERICAN LEGION HWY,766601.89,2933049.75,42.29569937610201,-71.09181948576261
4,5,2016-05-10 12:22:17+00,mv,Intersection,NaN,HARRISON AVE,NORTHAMPTON ST,770879.73,2947192.89,42.33446495632261,-71.07573628670127


,_id,date_time,mode_type,location_type,street,xstreet1,xstreet2,x_cord,y_cord,long,lat
0,1,2021-08-27 23:16:04+00,mv,Street,HUMBOLDT AVE,BROOKLEDGE ST,SEAVER ST,766700.610000000000,2938321.020000000000,-71.09175104914614,42.31035694234807
1,2,2020-09-06 20:59:05+00,mv,Intersection,NaN,W FIRST ST,E ST,778620.860000000000,2949151.140000000000,-71.04706795221954,42.33972922713294
2,3,2016-10-25 16:18:52+00,ped,Street,WASHINGTON ST,BEETHOVEN ST,SCHOOL ST,764638.340000000000,2940024.010000000000,-71.09911264638293,42.31496798346911
3,4,2024-10-04 23:16:10+00,ped,Intersection,NaN,WASHINGTON ST,BRAHMS ST,753957.700000000000,2926135.850000000000,-71.13865114944244,42.27689709027487
4,5,2024-04-01 00:36:18+00,mv,Intersection,NaN,KING ST,DORCHESTER AVE,774510.990000000000,2931681.860000000000,-71.06260056171318,42.291851640587666


## Normalize to the map schema

Records without a parseable event timestamp or earlier than 2021 are excluded. Coordinates are converted to numbers, and source IDs are kept as strings.

In [11]:
OUTPUT_COLUMNS = [
    "source", "id", "muni", "year", "date", "severity",
    "time", "police", "cyclist", "pedestrian", "other",
    "lat", "lng", "interstate", "datetime",
]

def normalize_records(
    raw: pd.DataFrame,
    *,
    timestamp_column: str,
    source_name: str,
    severity_value: int,
) -> pd.DataFrame:
    required = {"_id", "mode_type", "lat", "long", timestamp_column}
    missing = required.difference(raw.columns)
    if missing:
        raise ValueError(f"{source_name} data is missing required columns: {sorted(missing)}")

    parsed_timestamp = pd.to_datetime(raw[timestamp_column], errors="coerce")
    keep = parsed_timestamp.notna() & parsed_timestamp.dt.year.ge(START_YEAR)
    records = raw.loc[keep].copy()
    event_time = parsed_timestamp.loc[keep]
    mode = records["mode_type"].astype("string").str.strip().str.lower()

    normalized = pd.DataFrame(index=records.index)
    normalized["source"] = source_name
    normalized["id"] = records["_id"].astype("string")
    normalized["muni"] = "BOSTON"
    normalized["year"] = event_time.dt.year.astype("Int64")
    normalized["date"] = event_time.dt.strftime("%Y-%m-%d")
    normalized["severity"] = severity_value
    normalized["time"] = event_time.dt.strftime("%H:%M:%S")
    normalized["police"] = pd.NA
    normalized["cyclist"] = mode.eq("bike").astype("int8")
    normalized["pedestrian"] = mode.eq("ped").astype("int8")
    normalized["other"] = pd.NA
    normalized["lat"] = pd.to_numeric(records["lat"], errors="coerce")
    normalized["lng"] = pd.to_numeric(records["long"], errors="coerce")
    normalized["interstate"] = pd.NA
    normalized["datetime"] = event_time.dt.strftime("%Y-%m-%d %H:%M:%S")
    return normalized[OUTPUT_COLUMNS].reset_index(drop=True)

In [12]:
crash_clean = normalize_records(
    crash_raw,
    timestamp_column="dispatch_ts",
    source_name="BostonVZCrash",
    severity_value=2,
)
fatality_clean = normalize_records(
    fatality_raw,
    timestamp_column="date_time",
    source_name="BostonVZFatality",
    severity_value=1,
)

vision_zero_clean = pd.concat([crash_clean, fatality_clean], ignore_index=True)
vision_zero_clean = vision_zero_clean.sort_values("datetime", na_position="last").reset_index(drop=True)

assert vision_zero_clean.columns.tolist() == OUTPUT_COLUMNS
print(f"Clean Vision Zero records from {START_YEAR} onward: {len(vision_zero_clean):,}")
print(vision_zero_clean["source"].value_counts(dropna=False))
display(vision_zero_clean.head())

Clean Vision Zero records from 2021 onward: 19,684
source
BostonVZCrash       19612
BostonVZFatality       72
Name: count, dtype: int64


,source,id,muni,year,date,severity,time,police,cyclist,pedestrian,other,lat,lng,interstate,datetime
0,BostonVZCrash,37096,BOSTON,2021,2021-01-01,2,00:52:06,<NA>,0,0,<NA>,42.317431,-71.095713,<NA>,2021-01-01 00:52:06
1,BostonVZCrash,6897,BOSTON,2021,2021-01-01,2,01:24:34,<NA>,0,0,<NA>,42.283983,-71.090054,<NA>,2021-01-01 01:24:34
2,BostonVZCrash,32146,BOSTON,2021,2021-01-01,2,03:13:33,<NA>,0,0,<NA>,42.302084,-71.114459,<NA>,2021-01-01 03:13:33
3,BostonVZCrash,33123,BOSTON,2021,2021-01-01,2,03:43:35,<NA>,0,0,<NA>,42.330166,-71.056958,<NA>,2021-01-01 03:43:35
4,BostonVZCrash,30876,BOSTON,2021,2021-01-01,2,10:27:57,<NA>,0,0,<NA>,42.246395,-71.136270,<NA>,2021-01-01 10:27:57


In [13]:
vision_zero_clean.to_csv(OUTPUT_FILE, index=False)
print(f"Saved {len(vision_zero_clean):,} records to {OUTPUT_FILE.resolve()}")

Saved 19,684 records to C:\Users\johnbest\mass-crash-map\data\boston-vision-zero.csv
